In [ ]:
import numpy as np
import pandas as pd
class Node:
    def __init__(self,gini_gain=None,feature=None,best_value=None,left=None,right=None,X=None,y=None,majority_class=None):
        self.X=X
        self.y=y
        self.gini_gain=gini_gain
        self.best_feature=feature
        self.best_value=best_value
        self.left=left
        self.right=right
        self.majority_class=majority_class

class MyDecisionTree:
    def __init__(self):
        self.max_depth = None
        self.root=None
    
    def fit(self, X, y):
        self.tree = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        num_samples, num_features = X.shape
        unique_classes, counts = np.unique(y, return_counts=True)
        majority_class = unique_classes[np.argmax(counts)]

        if depth == self.max_depth or len(unique_classes) == 1 or num_samples < 2:
            return majority_class

        best_feature, best_value = self.find_best_split(X,y)
        if best_feature is None:
            return majority_class

        left_mask = [i for i in len(X) if X[i] <= best_value]
        right_mask = [i for i in len(X) if X[i] > best_value]

        left_subtree = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_subtree = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return Node(best_value, majority_class, left_subtree, right_subtree, X, y)
    
          
    
    def _gini_impurity(self, num_samples_per_class):
        total_samples = np.sum(num_samples_per_class)
        impurity = 1.0
        for num in num_samples_per_class:
            p = num / total_samples
            impurity -= p ** 2
        return impurity
    
    def calculate_gini_gain(self, current_impurity, num_samples_parent, num_samples_left, num_samples_right):
        p_left = np.sum(num_samples_left) / np.sum(num_samples_parent)
        p_right = np.sum(num_samples_right) / np.sum(num_samples_parent)
        return current_impurity - p_left * self._gini_impurity(num_samples_left) - p_right * self._gini_impurity(num_samples_right)

    def find_best_split(self,node):
        X=node.X
        num_samples, num_features = X.shape
        best_value=None
        best_feature=None
        best_gain=-2
        for index in num_features:
            feature=X[:,index]
            values=np.unique(feature)
            for value in values:
                left_mask = [i for i in len(X) if X[i] <= value]
                right_mask = [i for i in len(X) if X[i] > value]

                if(len(left_mask)==0 or len(right_mask)==0):
                    continue
                current=node.gini_gain
                gini_gain=self.calculate_gini_gain(current,y[left_mask],y[right_mask])
                if gini_gain>best_gain:
                    best_gain=gini_gain
                    best_feature=index
                    best_value=value

        return best_feature,best_value 
   


    # Create and fit the Decision Tree
    tree = MyDecisionTree(max_depth=3)
    tree.fit(X, y)

    # Predict and calculate accuracy
    accuracy = tree.score(X, y)
    print("Accuracy:", accuracy)


In [20]:
#without pruning
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
data = pd.read_csv('thyroid.csv')
#preprocessing
data=data[data['sex'] != '0']
data=data[data['label'] != 'goitre']
data=data[data['label'] != 'T3 toxic']

categorical_columns = data.select_dtypes(include=['object']).columns
print(categorical_columns)
data = pd.get_dummies(data, columns=categorical_columns,drop_first=True)
display(data)


Index(['sex', 'on thyroxine', 'query on thyroxine',
       'on antithyroid medication', 'sick', 'pregnant', 'thyroid surgery',
       'I131 treatment', 'query hypothyroid', 'query hyperthyroid', 'lithium',
       'goitre', 'tumor', 'hypopituitary', 'psych', 'TSH measured',
       'T3 measured', 'TT4 measured', 'T4U measured', 'FTI measured',
       'TBG measured', 'referral source', 'label'],
      dtype='object')


,age,TSH,T3,TT4,T4U,FTI,TBG,sex_M,on thyroxine_t,query on thyroxine_t,...,TSH measured_t,T3 measured_t,TT4 measured_t,T4U measured_t,FTI measured_t,referral source_SVHC,referral source_SVHD,referral source_SVI,referral source_other,label_negative
0,41,1.30,2.5,125.0,1.14,109.0,0,0,0,0,...,1,1,1,1,1,1,0,0,0,1
1,23,4.10,2.0,102.0,0.00,0.0,0,0,0,0,...,1,1,1,0,0,0,0,0,1,1
2,46,0.98,0.0,109.0,0.91,120.0,0,1,0,0,...,1,0,1,1,1,0,0,0,1,1
3,70,0.16,1.9,175.0,0.00,0.0,0,0,1,0,...,1,1,1,0,0,0,0,0,1,1
4,70,0.72,1.2,61.0,0.87,70.0,0,0,0,0,...,1,1,1,1,1,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2795,70,2.70,0.0,155.0,1.05,148.0,0,1,0,0,...,1,0,1,1,1,0,0,1,0,1
2796,73,0.00,0.7,63.0,0.88,72.0,0,1,0,1,...,0,1,1,1,1,0,0,0,1,1
2797,75,0.00,0.0,147.0,0.80,183.0,0,1,0,0,...,0,0,1,1,1,0,0,0,1,1
2798,60,1.40,0.0,100.0,0.83,121.0,0,0,0,0,...,1,0,1,1,1,0,0,0,1,1


In [37]:

class Node:
    def __init__(self, depth, impurity, feature=None, value=None, left=None, right=None, label=None):
        self.depth = depth
        self.impurity = impurity
        self.feature = feature
        self.value = value
        self.left = left
        self.right = right
        self.label = label

class MyDecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

    def gini_impurity(self, y):
        # Calculate Gini impurity of a node
        p1 = (y == 1).sum() / len(y)
        p2 = (y == 0).sum() / len(y)
        return 1 - p1**2 - p2**2
    
    def max_depth(self, depth):
        # Function to set the maximum depth of the tree
        self.max_depth = depth
    
    def cost_function(self, y, left_y, right_y):
        # Calculate Gini Gain
        m = len(y)
        gini_before = self.gini_impurity(y)
        m_left, m_right = len(left_y), len(right_y)
        gini_after = (m_left / m) * self.gini_impurity(left_y) + (m_right / m) * self.gini_impurity(right_y)
        return gini_before - gini_after

    def split_data(self, X, y, feature, value):
        # Split data into left and right based on a feature and value
        left_mask = X[:, feature] <= value
        right_mask = X[:, feature] > value
        left_X, left_y = X[left_mask], y[left_mask]
        right_X, right_y = X[right_mask], y[right_mask]
        return left_X, left_y, right_X, right_y

    def find_best_split(self, X, y):
        # Find the best feature and value to split on based on Gini Gain
        m, n = X.shape
        if m <= self.min_samples_split:
            return None, None, None, None

        current_impurity = self.gini_impurity(y)
        best_impurity = -1.0  # Initialize with a negative value
        best_feature = None
        best_value = None

        for feature in range(n):
            values = np.unique(X[:, feature])
            for value in values:
                left_X, left_y, right_X, right_y = self.split_data(X, y, feature, value)
                if len(left_y) == 0 or len(right_y) == 0:
                    continue
                gain = self.cost_function(y, left_y, right_y)
                if gain > best_impurity:
                    best_impurity = gain
                    best_feature = feature
                    best_value = value

        return best_feature, best_value, best_impurity, current_impurity

    def build_tree(self, X, y, depth=0):
        if depth == self.max_depth or self.gini_impurity(y) == 0:
            return Node(depth, self.gini_impurity(y), label=np.argmax(np.bincount(y)))

        best_feature, best_value, best_impurity, current_impurity = self.find_best_split(X, y)

        if best_impurity > 0:
            left_X, left_y, right_X, right_y = self.split_data(X, y, best_feature, best_value)
            left_subtree = self.build_tree(left_X, left_y, depth + 1)
            right_subtree = self.build_tree(right_X, right_y, depth + 1)
            return Node(depth, best_impurity, feature=best_feature, value=best_value, left=left_subtree, right=right_subtree)
        else:
            return Node(depth, current_impurity, label=np.argmax(np.bincount(y)))

    def fit(self, X, y):
        self.tree = self.build_tree(X, y)

    def predict_instance(self, instance, node):
        if node.label is not None:
            return node.label
        if instance[node.feature] <= node.value:
            return self.predict_instance(instance, node.left)
        else:
            return self.predict_instance(instance, node.right)

    def predict(self, X):
        return [self.predict_instance(instance, self.tree) for instance in X]

    def score(self, X, y):
        predictions = self.predict(X)
        return np.mean(predictions == y)

X=data.iloc[:,:-1]
y=data.iloc[:,-1]
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size= 0.2,random_state=0)

# Initialize and fit the Decision Tree
tree = MyDecisionTree(max_depth=5)  # Adjust max_depth as needed
tree.fit(X_train.values, y_train.values)

# Evaluate the Decision Tree
# y_test_pred = tree.predict(X_test.values)
accuracy = tree.score(X_test.values, y_test.values)
print("Accuracy:", accuracy)


Accuracy: 0.9869402985074627


In [39]:
#decision tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,classification_report

X=data.iloc[:,:-1]
y=data.iloc[:,-1]
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size= 0.2,random_state=0)
tree_entropy = DecisionTreeClassifier(criterion='entropy')
tree_gini = DecisionTreeClassifier(criterion='gini')

# Fit the classifiers on the training data
tree_entropy.fit(X_train, y_train)
tree_gini.fit(X_train, y_train)

# Predict using both classifiers
y_pred_entropy = tree_entropy.predict(X_test)
y_pred_gini = tree_gini.predict(X_test)

# Calculate accuracy scores for both classifiers
accuracy_entropy = accuracy_score(y_test, y_pred_entropy)
accuracy_gini = accuracy_score(y_test, y_pred_gini)

print('Accuracy of decision tree with entropy criterion: ', accuracy_entropy)
print('Accuracy of decision tree with gini criterion: ', accuracy_gini)

Accuracy of decision tree with entropy criterion:  0.9906716417910447
Accuracy of decision tree with gini criterion:  0.9888059701492538
